# Which function is missing? — reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alestainer/statistics-intuitions/blob/main/notebooks/04-support-boundary.ipynb)

Reproduces the 20 fixed support-boundary questions, exact prompts and saved model decisions used in the article. No paid request runs automatically.


In [ ]:
from pathlib import Path
import json, urllib.request

RAW = "https://raw.githubusercontent.com/Alestainer/statistics-intuitions/main/"

def load_json(relative_path):
    candidates = [Path("../") / relative_path, Path(relative_path)]
    for path in candidates:
        if path.exists():
            return json.loads(path.read_text())
    with urllib.request.urlopen(RAW + relative_path) as response:
        return json.load(response)

benchmark = load_json("data/04-support-boundary/benchmark.json")
results = load_json("data/04-support-boundary/results.json")
print(f"{len(benchmark['episodes'])} questions; {len(results['models'])} completed model runs")


## Inspect the exact images and options

In [ ]:
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

def read_bytes(relative_path):
    for path in (Path("../") / relative_path, Path(relative_path)):
        if path.exists(): return path.read_bytes()
    with urllib.request.urlopen(RAW + relative_path) as response: return response.read()

def episode_image(episode):
    name = Path(episode["image"]).name
    return Image.open(BytesIO(read_bytes("data/04-support-boundary/stimuli/" + name))).convert("RGB")

episode = benchmark["episodes"][0]
plt.figure(figsize=(10, 6)); plt.imshow(episode_image(episode)); plt.axis("off"); plt.show()
print(episode["userPrompt"])
print("Expected:", episode["answer"])


## Verify deterministic generation

In [ ]:
import math

RECT = {"xMin": -20., "xMax": 20., "yMin": -20., "yMax": 20.}

def hash_string(value):
    result = 2166136261
    for char in value:
        result ^= ord(char); result = (result * 16777619) & 0xffffffff
    return result

class Mulberry32:
    def __init__(self, seed): self.state = seed & 0xffffffff
    def __call__(self):
        self.state = (self.state + 0x6D2B79F5) & 0xffffffff
        value = self.state
        value = ((value ^ (value >> 15)) * (value | 1)) & 0xffffffff
        value ^= (value + (((value ^ (value >> 7)) * (value | 61)) & 0xffffffff)) & 0xffffffff
        return ((value ^ (value >> 14)) & 0xffffffff) / 4294967296

FUNCTIONS = {
    "line": lambda x: x, "negative-line": lambda x: -3*x,
    "parabola": lambda x: x*x, "sine": lambda x: 10*math.sin(x),
    "exp-sine": lambda x: math.exp(x/7)*math.sin(x),
    "log": lambda x: math.log(x) if x > 0 else math.nan,
    "exponential": lambda x: 2**x,
}

def is_above(function_id, x, y):
    return x < math.exp(y) if function_id == "log" else y > FUNCTIONS[function_id](x)

def regenerate_points(episode):
    rng = Mulberry32(hash_string(f'{episode["seed"]}:{episode["pointCount"]}'))
    function_ids = [f["id"] for f in benchmark["generation"]["functions"]]
    hidden = function_ids[math.floor(rng() * len(function_ids))]
    side = "above" if rng() < .5 else "below"
    points = []
    while len(points) < episode["pointCount"]:
        x = RECT["xMin"] + rng() * 40; y = RECT["yMin"] + rng() * 40
        above = is_above(hidden, x, y)
        if (side == "above" and above) or (side == "below" and not above): points.append((x, y))
    return hidden, side, points

hidden, side, points = regenerate_points(episode)
assert hidden == episode["hiddenFunctionId"] and side == episode["side"] and len(points) == 200
print("Regenerated:", hidden, side, len(points), "points")


## Exact model prompt

In [ ]:
print(benchmark["systemPrompt"])
print("\n--- example user message ---\n")
print(episode["userPrompt"])


## Recompute the published table

In [ ]:
import pandas as pd

rows = []
for run in results["models"]:
    episodes = run["episodes"]
    rows.append({
        "model": run["model"],
        "correct": f'{sum(e["correct"] for e in episodes)}/20',
        "invalid responses": sum(not e["strictlyFormatted"] for e in episodes),
        "cost (USD)": round(sum(e["costUsd"] for e in episodes), 4),
    })
pd.DataFrame(rows).sort_values("correct", ascending=False).reset_index(drop=True)


## Optional paid rerun

To rerun a question, send `benchmark["systemPrompt"]` as the system message and the episode's `userPrompt` plus `episode_image(episode)` as the user message. Accept only an exact `OPTION A`, `OPTION B`, `OPTION C` or `OPTION D`. API calls are not part of automatic notebook execution because providers and prices can change.
